# Notebook 00 — Data Inspection

Quick look at the LMP and regulation-price series we will use in the rest of the project.
If a real PJM CSV is present in `data/pjm/rt_lmps.csv`, it is used; otherwise we fall back to the synthetic generator and label everything as such.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.utils.pjm_data_loader import (
    align_lmp_and_reg, find_real_or_synthetic, load_reg_prices, load_rt_lmps,
)
from src.utils.synthetic_data import generate_synthetic_dataset

sns.set_theme(style='whitegrid')
using_real, label = find_real_or_synthetic('../data/pjm')
print(f'Data source: {label}')

In [ ]:
if using_real:
    lmps = load_rt_lmps('../data/pjm/rt_lmps.csv', pnode_id=51217)
    reg = load_reg_prices('../data/pjm/reg_prices.csv')
    df = align_lmp_and_reg(lmps, reg)
else:
    df = generate_synthetic_dataset(n_days=14, seed=42)
df.head()

## Plot one week of LMP

In [ ]:
et = df.index.tz_convert('America/New_York')
week = df.iloc[:7*24*12]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(week.index.tz_convert('America/New_York'), week['lmp'], linewidth=0.9)
ax.set_xlabel('Time (ET)'); ax.set_ylabel('LMP ($/MWh)')
ax.set_title(f'One week of LMP — {label}')
plt.show()

## LMP distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.histplot(df['lmp'], bins=60, ax=ax, color='C0')
ax.set_xlabel('LMP ($/MWh)'); ax.set_title('LMP distribution')
plt.show()

## Regulation prices

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(week.index.tz_convert('America/New_York'), week['reg_cap_price'], label='Reg cap', linewidth=0.9)
ax.plot(week.index.tz_convert('America/New_York'), week['reg_perf_price'], label='Reg perf', linewidth=0.9)
ax.set_xlabel('Time (ET)'); ax.set_ylabel('Reg price ($/MW-h)'); ax.legend()
ax.set_title(f'Regulation prices — {label}')
plt.show()

## Summary statistics

In [ ]:
df.describe().T